# 🚀 Model Training & Fine-Tuning with LocalTrack & Hugging Face

This notebook demonstrates how to train or fine-tune machine learning models with PyTorch and Hugging Face `transformers` while streaming live metrics directly to **LocalTrack**.

### Features:
- Automatic hyperparameter capture (`learning_rate`, `batch_size`, `warmup_steps`, optimizer)
- High-throughput asynchronous metric streaming (`train/loss`, `eval/accuracy`, `learning_rate`)
- Artifact uploads (`config.json`, `trainer_state.json`)

In [ ]:
# 1. Imports and environment setup
import math
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from backend.sdk.localtrack import LocalTrackClient, LocalTrackHFCallback

print(f"PyTorch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

## 2. Initialize LocalTrack Logger Callback

Configure the `LocalTrackHFCallback` to track the experiment under the `llm-finetuning` project.

In [ ]:
# Create LocalTrack Hugging Face Callback
callback = LocalTrackHFCallback(
    project_name="llm-finetuning",
    run_name="llama3-lora-r16",
    base_url="http://127.0.0.1:8000",
    tags=["llama-3", "lora", "sft"],
)

print(f"Configured LocalTrack callback for project: {callback.project_name}")

## 3. Simulated Training Loop with Metric Logging

For environments without heavy GPUs, here is the simulated high-speed training loop capturing the exact telemetry emitted by `Trainer`.

In [ ]:
# Direct Client Logging Demonstration
client = LocalTrackClient(base_url="http://127.0.0.1:8000")
run_id = client.init_run(
    project_name="llm-finetuning",
    run_name="llama3-lora-r16",
    config={
        "learning_rate": 2e-4,
        "batch_size": 16,
        "lora_rank": 16,
        "lora_alpha": 32,
        "epochs": 3,
    },
    tags=["llama-3", "lora", "sft"],
)

print(f"Started LocalTrack Run ID: {run_id}")

total_steps = 100
for step in range(1, total_steps + 1):
    loss = 2.5 * math.exp(-step / 30.0) + random.uniform(-0.05, 0.05) + 0.35
    lr = 2e-4 * (1.0 - step / total_steps)
    client.log_metrics({
        "train/loss": round(loss, 4),
        "train/learning_rate": lr,
        "train/epoch": round(step / 33.3, 2),
    }, step=step)

client.finish_run(status="finished")
print("✅ Training complete! Open LocalTrack Workspace to view the live curves.")